# Phi model comparison

Run this notebook in a fresh Colab runtime. It keeps Phi in its own environment, checkpoints every batch to Google Drive, and resumes completed cases after a disconnect.

In [1]:
!pip -q install 'transformers==4.49.0' 'accelerate==1.3.0' bitsandbytes 'mlflow==3.15.1'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 124.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.6/336.6 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 134.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 129.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import gc
import json
import os
import time
from importlib.metadata import version
from pathlib import Path

import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from src.schema import ALLOWED_DOMAINS, ALLOWED_ISSUES, SchemaError, validate_gold

assert torch.cuda.is_available(), 'Select a Colab GPU runtime before running.'
drive.mount('/content/drive')
print('torch:', torch.__version__)
print('transformers:', version('transformers'))
print('gpu:', torch.cuda.get_device_name(0))

MODEL_NAME = 'microsoft/Phi-4-mini-instruct'
MODEL_REVISION = 'main'
MAX_NEW_TOKENS = 192
BATCH_SIZE = 4
CHECKPOINT_ROOT = Path('/content/drive/MyDrive/civicstruct-bakeoff')
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

Mounted at /content/drive
torch: 2.11.0+cu128
transformers: 4.49.0
gpu: Tesla T4


In [3]:
data_paths = [Path('data/model_selection_development.jsonl'), Path('../data/model_selection_development.jsonl')]
data_path = next(path for path in data_paths if path.exists())
development = [json.loads(line) for line in data_path.read_text(encoding='utf-8').splitlines() if line.strip()]
assert len(development) == 40
for row in development:
    validate_gold(row['gold'])
demo_paths = [Path('data/gold_examples.jsonl'), Path('../data/gold_examples.jsonl')]
demo_path = next(path for path in demo_paths if path.exists())
static_demos = [json.loads(line) for line in demo_path.read_text(encoding='utf-8').splitlines() if line.strip()]
assert not {row['case_id'] for row in development} & {row['case_id'] for row in static_demos}
print('development complaints:', len(development))
print('fixed demonstrations:', len(static_demos))

development complaints: 40
fixed demonstrations: 5


In [4]:
SYSTEM_PROMPT = (
    'You structure public-service complaints. Return exactly one JSON object with these fields: '
    'service_domain, issue_type, location, event_date_or_time, amount_inr, service_identifier, '
    'urgency, missing_information, and formal_summary. Use null for an absent scalar fact. '
    'Use only the allowed labels from the task schema. Do not guess facts. Do not output reasoning or commentary.'
)

def build_messages(complaint, few_shot=False):
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    if few_shot:
        for demo in static_demos:
            messages.extend([
                {'role': 'user', 'content': demo['complaint']},
                {'role': 'assistant', 'content': json.dumps(demo['gold'], separators=(',', ':'))},
            ])
    messages.append({'role': 'user', 'content': complaint})
    return messages

def save_json(path, value):
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2), encoding='utf-8')
    temporary.replace(path)

def load_json(path, default):
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else default

def prepare_inputs(tokenizer, message_batch):
    args = {'tokenize': True, 'add_generation_prompt': True, 'return_dict': True, 'return_tensors': 'pt', 'padding': True}
    return tokenizer.apply_chat_template(message_batch, **args)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, trust_remote_code=True)
tokenizer.padding_side = 'left'
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    trust_remote_code=True,
    quantization_config=quantization_config,
    device_map='auto',
)
model.eval()
print('clean reload:', MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.77G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

clean reload: microsoft/Phi-4-mini-instruct


In [5]:
def generate_batch(rows, few_shot):
    inputs = prepare_inputs(tokenizer, [build_messages(row['complaint'], few_shot) for row in rows]).to(next(model.parameters()).device)
    prompt_tokens = inputs['input_ids'].shape[-1]
    torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=MAX_NEW_TOKENS,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    torch.cuda.synchronize()
    per_case = (time.perf_counter() - started) / len(rows)
    responses = tokenizer.batch_decode(generated[:, prompt_tokens:], skip_special_tokens=True)
    return [
        {'case_id': row['case_id'], 'response': response.strip(), 'latency_seconds': round(per_case, 4), 'batch_size': len(rows)}
        for row, response in zip(rows, responses)
    ]

def generate_with_fallback(rows, few_shot):
    try:
        return generate_batch(rows, few_shot)
    except RuntimeError as error:
        if 'out of memory' not in str(error).lower() or len(rows) == 1:
            raise
        torch.cuda.empty_cache()
        return [item for row in rows for item in generate_batch([row], few_shot)]

def macro_f1(gold_values, predicted_values, labels):
    scores = []
    for label in labels:
        tp = sum(gold == label and predicted == label for gold, predicted in zip(gold_values, predicted_values))
        fp = sum(gold != label and predicted == label for gold, predicted in zip(gold_values, predicted_values))
        fn = sum(gold == label and predicted != label for gold, predicted in zip(gold_values, predicted_values))
        denominator = 2 * tp + fp + fn
        scores.append(0.0 if denominator == 0 else 2 * tp / denominator)
    return sum(scores) / len(scores)

def score_outputs(outputs):
    valid = 0
    gold_domains, predicted_domains = [], []
    gold_issues, predicted_issues = [], []
    missing_parts = []
    hallucinated = filled = 0
    for row, output in zip(development, outputs):
        try:
            predicted = json.loads(output['response'])
            validate_gold(predicted)
            valid += 1
        except (json.JSONDecodeError, SchemaError, TypeError, ValueError):
            predicted = {}
        gold = row['gold']
        gold_domains.append(gold['service_domain'])
        predicted_domains.append(predicted.get('service_domain'))
        gold_issues.append(gold['issue_type'])
        predicted_issues.append(predicted.get('issue_type'))
        missing_parts.append((set(gold['missing_information']), set(predicted.get('missing_information') or [])))
        complaint = row['complaint'].lower().replace(',', '')
        for field in ('location', 'event_date_or_time', 'service_identifier', 'amount_inr'):
            value = predicted.get(field)
            if value is not None:
                filled += 1
                hallucinated += str(value).lower().replace(',', '') not in complaint
    tp = sum(len(gold & predicted) for gold, predicted in missing_parts)
    fp = sum(len(predicted - gold) for gold, predicted in missing_parts)
    fn = sum(len(gold - predicted) for gold, predicted in missing_parts)
    denominator = 2 * tp + fp + fn
    return {
        'schema_validity_rate': valid / len(outputs),
        'service_domain_macro_f1': macro_f1(gold_domains, predicted_domains, ALLOWED_DOMAINS),
        'issue_type_macro_f1': macro_f1(gold_issues, predicted_issues, ALLOWED_ISSUES),
        'missing_information_f1': 0.0 if denominator == 0 else 2 * tp / denominator,
        'hallucinated_field_rate': 0.0 if filled == 0 else hallucinated / filled,
    }

In [8]:
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
import mlflow
MLFLOW_ROOT = CHECKPOINT_ROOT / 'mlruns'
MLFLOW_ROOT.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(MLFLOW_ROOT.as_uri())
mlflow.set_experiment('civicstruct-model-bakeoff')
summary_path = CHECKPOINT_ROOT / 'phi_bakeoff_summary.json'
summary = load_json(summary_path, {'results': [], 'setup_failures': []})
all_results = summary['results']
for prompt_mode in ('zero_shot', 'static_few_shot'):
    if any(item['prompt_mode'] == prompt_mode for item in all_results):
        print('already complete:', prompt_mode)
        continue
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    checkpoint_path = CHECKPOINT_ROOT / f'phi_4_mini__{prompt_mode}.json'
    checkpoint = load_json(checkpoint_path, {'outputs': []})
    outputs_by_id = {item['case_id']: item for item in checkpoint['outputs']}
    remaining = [row for row in development if row['case_id'] not in outputs_by_id]
    for start in range(0, len(remaining), BATCH_SIZE):
        batch = remaining[start:start + BATCH_SIZE]
        generated = generate_with_fallback(batch,prompt_mode == 'static_few_shot',)
        outputs_by_id.update({item['case_id']: item for item in generated})
        ordered = [outputs_by_id[row['case_id']] for row in development if row['case_id'] in outputs_by_id]
        save_json(checkpoint_path, {'model_name': MODEL_NAME, 'prompt_mode': prompt_mode, 'model_revision': MODEL_REVISION, 'batch_size': BATCH_SIZE, 'max_new_tokens': MAX_NEW_TOKENS, 'outputs': ordered})
        print(f'{prompt_mode}: saved {len(ordered)}/{len(development)}')
    outputs = [outputs_by_id[row['case_id']] for row in development]
    scores = score_outputs(outputs)
    result = {
        'model_name': MODEL_NAME,
        'model_revision': getattr(model.config, '_commit_hash', None) or MODEL_REVISION,
        'prompt_mode': prompt_mode,
        'device': 'cuda',
        'decoding': {'do_sample': False, 'max_new_tokens': MAX_NEW_TOKENS},
        'batch_size': BATCH_SIZE,
        'mean_latency_seconds_per_case': sum(item['latency_seconds'] for item in outputs) / len(outputs),
        'gpu_memory_mb': torch.cuda.max_memory_allocated() / 2**20,
        'package_versions': {name: version(name) for name in ('torch', 'transformers', 'mlflow')},
        'scores': scores,
    }
    try:
        with mlflow.start_run(run_name='phi-4-mini-' + prompt_mode) as run:
            mlflow.log_params({'model_name': MODEL_NAME, 'model_revision': result['model_revision'], 'prompt_mode': prompt_mode, 'decoding': 'greedy', 'batch_size': BATCH_SIZE})
            mlflow.log_metrics({**scores, 'mean_latency_seconds_per_case': result['mean_latency_seconds_per_case'], 'gpu_memory_mb': result['gpu_memory_mb']})
            mlflow.log_text(json.dumps(result, indent=2), 'run_metadata.json')
            mlflow.log_text(json.dumps(outputs, indent=2), 'outputs.json')
            result['mlflow_run_id'] = run.info.run_id
    except Exception as error:
        result['mlflow_error'] = f'{type(error).__name__}: {error}'
    all_results = [item for item in all_results if item['prompt_mode'] != prompt_mode] + [result]
    save_json(summary_path, {'results': all_results, 'setup_failures': []})
    print(json.dumps(result, indent=2))

del tokenizer, model
gc.collect()
torch.cuda.empty_cache()
print(json.dumps({'results': all_results}, indent=2))

zero_shot: saved 4/40
zero_shot: saved 8/40
zero_shot: saved 12/40
zero_shot: saved 16/40
zero_shot: saved 20/40
zero_shot: saved 24/40
zero_shot: saved 28/40
zero_shot: saved 32/40
zero_shot: saved 36/40
zero_shot: saved 40/40
{
  "model_name": "microsoft/Phi-4-mini-instruct",
  "model_revision": "cfbefacb99257ffa30c83adab238a50856ac3083",
  "prompt_mode": "zero_shot",
  "device": "cuda",
  "decoding": {
    "do_sample": false,
    "max_new_tokens": 192
  },
  "batch_size": 4,
  "mean_latency_seconds_per_case": 1.88951,
  "gpu_memory_mb": 3065.49560546875,
  "package_versions": {
    "torch": "2.11.0+cu128",
    "transformers": "4.49.0",
    "mlflow": "3.15.1"
  },
  "scores": {
    "schema_validity_rate": 0.0,
    "service_domain_macro_f1": 0.0,
    "issue_type_macro_f1": 0.0,
    "missing_information_f1": 0.0,
    "hallucinated_field_rate": 0.0
  },
  "mlflow_run_id": "efe2c988588c47898ea2c8fa58c9bfde"
}
static_few_shot: saved 4/40
static_few_shot: saved 8/40
static_few_shot: saved 